# SatDetection — Real-World Inference via Google Earth Engine
**Generalization test: run the trained model on unseen satellite imagery**

CSCI 4366/6366 — Neural Networks and Deep Learning, Spring 2026  
Team: Rufai Yakubu, Aaron Tyler

---
This notebook:
1. Pulls a cloud-free Sentinel-2 image of a new city from GEE
2. Exports it to Google Drive as a GeoTIFF
3. Tiles it into 256x256 patches
4. Runs the best trained checkpoint on every patch
5. Stitches predictions back into a full-scene map
6. Visualizes building and road predictions overlaid on the image

**No ground truth** — this is a visual generalization check only.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install earthengine-api segmentation-models-pytorch albumentations rasterio pyyaml tqdm

## 2. Authenticate Google Earth Engine
Requires a GEE account. Sign up free at https://earthengine.google.com/

In [ ]:
import ee
ee.Authenticate()   # opens browser auth flow
ee.Initialize(project='your-gee-project-id')  # <-- update with your GEE project ID

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

BASE_DIR    = '/content/drive/MyDrive/SatDetection'
GEE_DIR     = os.path.join(BASE_DIR, 'gee')
INFER_DIR   = os.path.join(BASE_DIR, 'gee/inference')

for d in [GEE_DIR, INFER_DIR,
          os.path.join(GEE_DIR, 'tiles/images'),
          os.path.join(GEE_DIR, 'tiles/masks'),
          os.path.join(INFER_DIR, 'visuals')]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted.')

## 4. Clone Repo

In [ ]:
REPO_URL = 'https://github.com/YOUR_USERNAME/SatDetection.git'  # <-- update this

if not os.path.exists('/content/SatDetection'):
    !git clone $REPO_URL /content/SatDetection
else:
    !git -C /content/SatDetection pull

sys.path.insert(0, '/content/SatDetection/src')
print('Repo ready.')

## 5. Choose a City

Pick a city the model has never seen — not Poland, not Vegas.

We use **Accra, Ghana** by default — dense urban area, mix of formal and informal settlements, very different from the training data. Change the coordinates to any city you like.

In [ ]:
# City options — uncomment the one you want

CITY_NAME = 'Accra_Ghana'
AOI = ee.Geometry.Rectangle([-0.255, 5.530, -0.150, 5.620])  # Accra, Ghana

# CITY_NAME = 'Nairobi_Kenya'
# AOI = ee.Geometry.Rectangle([36.780, -1.320, 36.880, -1.230])

# CITY_NAME = 'Houston_USA'
# AOI = ee.Geometry.Rectangle([-95.420, 29.720, -95.320, 29.800])

# CITY_NAME = 'Dubai_UAE'
# AOI = ee.Geometry.Rectangle([55.260, 25.180, 55.360, 25.260])

print(f'Selected city: {CITY_NAME}')

## 6. Pull Cloud-Free Sentinel-2 Image from GEE

Sentinel-2 provides 10m/pixel RGB imagery — freely available globally.  
We filter for low cloud cover and take the median composite.

In [ ]:
# Pull Sentinel-2 SR (surface reflectance) — cloud-free median composite
s2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(AOI)
    .filterDate('2023-01-01', '2024-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .select(['B4', 'B3', 'B2'])   # Red, Green, Blue
    .median()
    .clip(AOI)
)

# Sentinel-2 values are 0-10000 — scale to 0-255 uint8 for export
s2_rgb = s2.divide(10000).multiply(255).toUint8()

print(f'Sentinel-2 image ready for export.')
print(f'AOI: {AOI.getInfo()["coordinates"]}')

## 7. Export to Google Drive

This starts a GEE export task. It runs on Google's servers — check progress at https://code.earthengine.google.com/  
Typically takes **5–15 minutes**. Wait for it to complete before continuing.

In [ ]:
export_filename = f'sentinel2_{CITY_NAME}'
export_folder   = 'SatDetection/gee'

task = ee.batch.Export.image.toDrive(
    image       = s2_rgb,
    description = export_filename,
    folder      = export_folder,
    fileNamePrefix = export_filename,
    region      = AOI,
    scale       = 10,          # 10m/pixel (native Sentinel-2 resolution)
    maxPixels   = 1e9,
    fileFormat  = 'GeoTIFF',
)
task.start()

print(f'Export task started: {export_filename}')
print('Monitor at: https://code.earthengine.google.com/ → Tasks tab')
print(f'File will appear at: MyDrive/{export_folder}/{export_filename}.tif')

In [ ]:
# Poll export status — run this cell until status shows COMPLETED
import time

while True:
    status = task.status()
    state  = status['state']
    print(f'Export status: {state}')
    if state in ('COMPLETED', 'FAILED', 'CANCELLED'):
        break
    time.sleep(30)

if state == 'COMPLETED':
    print('Export complete. Proceed to next cell.')
else:
    print(f'Export ended with state: {state}. Check GEE Tasks tab.')

## 8. Tile the Exported Image

In [ ]:
import numpy as np
import rasterio
from rasterio.windows import Window

def tile_inference_image(image_path, out_dir, tile_size=256, stride=256):
    """
    Tile a GeoTIFF for inference — no masks, saves tiles and their (row, col) positions
    so predictions can be stitched back into a full scene.
    """
    os.makedirs(os.path.join(out_dir, 'tiles'), exist_ok=True)

    positions = []  # (row, col) for each tile — needed for stitching

    with rasterio.open(image_path) as src:
        height, width = src.height, src.width
        count = 0

        for row in range(0, height - tile_size + 1, stride):
            for col in range(0, width - tile_size + 1, stride):
                window     = Window(col, row, tile_size, tile_size)
                image_tile = src.read(window=window)  # (C, H, W)

                if image_tile.shape[1] != tile_size or image_tile.shape[2] != tile_size:
                    continue

                np.save(os.path.join(out_dir, 'tiles', f'tile_{count:05d}.npy'), image_tile)
                positions.append((row, col))
                count += 1

    np.save(os.path.join(out_dir, 'positions.npy'), np.array(positions))
    print(f'Tiled {count} patches from {os.path.basename(image_path)}')
    print(f'Image size: {height} x {width} px')
    return count, height, width


GEE_TIFF = os.path.join(GEE_DIR, f'sentinel2_{CITY_NAME}.tif')
TILE_DIR  = os.path.join(GEE_DIR, 'tiles')

assert os.path.exists(GEE_TIFF), f'GeoTIFF not found: {GEE_TIFF}. Has the export completed?'

n_tiles, IMG_H, IMG_W = tile_inference_image(GEE_TIFF, TILE_DIR, tile_size=256, stride=256)

## 9. Load Best Checkpoint
Use Model B (pretrained) — expected to be the best performer.

In [ ]:
import torch
from model import build_model

CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')
CHECKPOINT     = os.path.join(CHECKPOINT_DIR, 'pretrained_best.pth')  # Model B

assert os.path.exists(CHECKPOINT), f'Checkpoint not found: {CHECKPOINT}'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = build_model(pretrained=False)  # load architecture, weights come from checkpoint
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model  = model.to(device)
model.eval()

print(f'Model loaded from: {CHECKPOINT}')
print(f'Running on: {device}')

## 10. Run Inference on All Tiles

In [ ]:
import albumentations as A
from tqdm import tqdm

# Same normalization used during training (LandCover.ai stats)
normalize = A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

TILE_SIZE   = 256
tile_files  = sorted(os.listdir(os.path.join(TILE_DIR, 'tiles')))
all_preds   = []   # list of (2, H, W) predictions per tile

with torch.no_grad():
    for fname in tqdm(tile_files, desc='Inference'):
        tile = np.load(os.path.join(TILE_DIR, 'tiles', fname))  # (C, H, W)

        # preprocess
        img = np.transpose(tile[:3], (1, 2, 0)).astype(np.float32)
        if img.max() > 1.0:
            img = img / 255.0
        img = normalize(image=img)['image']
        img = torch.from_numpy(np.transpose(img, (2, 0, 1))).unsqueeze(0).float().to(device)

        pred = model(img).squeeze(0).cpu().numpy()  # (2, H, W)
        all_preds.append(pred)

print(f'Inference complete on {len(all_preds)} tiles.')

## 11. Stitch Predictions into Full-Scene Map

In [ ]:
positions = np.load(os.path.join(TILE_DIR, 'positions.npy'))  # (N, 2) row/col

# build empty canvases for buildings and roads
pred_building_map = np.zeros((IMG_H, IMG_W), dtype=np.float32)
pred_road_map     = np.zeros((IMG_H, IMG_W), dtype=np.float32)

for (row, col), pred in zip(positions, all_preds):
    pred_building_map[row:row + TILE_SIZE, col:col + TILE_SIZE] = pred[0]
    pred_road_map[row:row + TILE_SIZE,     col:col + TILE_SIZE] = pred[1]

# save raw probability maps
np.save(os.path.join(INFER_DIR, f'{CITY_NAME}_buildings.npy'), pred_building_map)
np.save(os.path.join(INFER_DIR, f'{CITY_NAME}_roads.npy'),     pred_road_map)

print(f'Prediction maps stitched: {pred_building_map.shape}')
print(f'Building coverage: {(pred_building_map > 0.5).mean() * 100:.1f}% of pixels')
print(f'Road coverage:     {(pred_road_map > 0.5).mean() * 100:.1f}% of pixels')

## 12. Visualize Full-Scene Prediction

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio

THRESHOLD = 0.5

# load original full image for background
with rasterio.open(GEE_TIFF) as src:
    full_image = src.read()  # (C, H, W)

rgb = np.transpose(full_image[:3], (1, 2, 0)).astype(np.float32)
if rgb.max() > 1.0:
    rgb = rgb / 255.0
rgb = np.clip(rgb, 0, 1)

building_bin = pred_building_map > THRESHOLD
road_bin     = pred_road_map     > THRESHOLD

overlay = rgb.copy()
overlay[building_bin] = [1.0, 0.3, 0.3]   # red = buildings
overlay[road_bin]     = [0.3, 0.3, 1.0]   # blue = roads

fig, axes = plt.subplots(1, 3, figsize=(20, 8))

axes[0].imshow(rgb)
axes[0].set_title(f'Sentinel-2 RGB — {CITY_NAME.replace("_", ", ")}', fontsize=13)
axes[0].axis('off')

axes[1].imshow(overlay)
axes[1].set_title('Predicted: Buildings (red) + Roads (blue)', fontsize=13)
axes[1].axis('off')

# side-by-side probability heatmaps
combined = np.zeros((*pred_building_map.shape, 3))
combined[:, :, 0] = pred_building_map   # red channel = building confidence
combined[:, :, 2] = pred_road_map       # blue channel = road confidence
axes[2].imshow(combined)
axes[2].set_title('Confidence Map (red=buildings, blue=roads)', fontsize=13)
axes[2].axis('off')

building_patch = mpatches.Patch(color=(1.0, 0.3, 0.3), label='Buildings')
road_patch     = mpatches.Patch(color=(0.3, 0.3, 1.0), label='Roads')
fig.legend(handles=[building_patch, road_patch], loc='lower center', ncol=2, fontsize=12)

plt.suptitle(f'SatDetection — Generalization Test: {CITY_NAME.replace("_", ", ")}', fontsize=15)
plt.tight_layout()

save_path = os.path.join(INFER_DIR, 'visuals', f'{CITY_NAME}_full_scene.png')
plt.savefig(save_path, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved: {save_path}')

## 13. Tile-Level Spot Checks
Look at individual tiles for closer inspection.

In [ ]:
import random

N_SAMPLES  = 6
indices    = random.sample(range(len(all_preds)), N_SAMPLES)

fig, axes = plt.subplots(N_SAMPLES, 3, figsize=(12, N_SAMPLES * 4))
col_titles = ['Input Tile', 'Building Prediction', 'Road Prediction']

for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=11)

for i, idx in enumerate(indices):
    tile = np.load(os.path.join(TILE_DIR, 'tiles', tile_files[idx]))
    pred = all_preds[idx]

    rgb_tile = np.transpose(tile[:3], (1, 2, 0)).astype(np.float32)
    if rgb_tile.max() > 1.0:
        rgb_tile = rgb_tile / 255.0
    rgb_tile = np.clip(rgb_tile, 0, 1)

    axes[i][0].imshow(rgb_tile)
    axes[i][0].axis('off')

    axes[i][1].imshow(pred[0], cmap='Reds',  vmin=0, vmax=1)
    axes[i][1].axis('off')

    axes[i][2].imshow(pred[1], cmap='Blues', vmin=0, vmax=1)
    axes[i][2].axis('off')

plt.suptitle(f'Tile-Level Spot Checks — {CITY_NAME.replace("_", ", ")}', fontsize=13)
plt.tight_layout()

save_path = os.path.join(INFER_DIR, 'visuals', f'{CITY_NAME}_spot_checks.png')
plt.savefig(save_path, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved: {save_path}')

## 14. Notes on Interpretation

When reviewing results keep in mind:

- **Domain gap**: model was trained on aerial imagery (25cm/px, Poland). Sentinel-2 is satellite at 10m/px — buildings look much smaller and roads thinner at this resolution.
- **Expected behaviour**: building detection may be patchy on smaller structures. Road detection may pick up major highways but miss small streets.
- **This is a visual check only** — no IoU can be computed without ground truth.
- If predictions look reasonable (buildings roughly where buildings are, roads roughly along road networks), the model has generalized meaningfully across sensor and geography.
- If predictions are mostly noise, the domain gap between aerial (training) and satellite (inference) is too large — the model needs fine-tuning on Sentinel-2 data.